# Аудит журнала изменений моделей

Источник: `t_ent_param_chg`. Ноутбук определяет фактические типы сущностей, качество журнала, возможность связи с версиями моделей и пригодность для риск-контролей.

Персональные данные не выводятся и не записываются: автор изменения представлен только псевдонимным ключом `xxhash64`. Все вычисления выполняются Spark SQL/DataFrame API без Python UDF и без создания DataFrame из локальных Python-объектов.

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.storagelevel import StorageLevel

SOURCE_DB = 'prx_pri_custom_ris_l_library_custom_risk_model_library'
CHANGE_TABLE = SOURCE_DB + '.t_ent_param_chg'
MODEL_VER_TABLE = SOURCE_DB + '.t_model_ver'
VALID_TABLE = SOURCE_DB + '.t_valid'

OUTPUT_DB = 'arnsdpsbx_t_team_oam_sva_2'
OUTPUT_PROFILE = OUTPUT_DB + '.pri_model_change_log_profile'
OUTPUT_DETAILS = OUTPUT_DB + '.pri_model_change_log_details'
WRITE_RESULTS = False

RECENT_DAYS = 30
POST_VALIDATION_HOURS = 24
BURST_CHANGES_PER_HOUR = 10
CRITICAL_PARAM_RX = r'(?i)(status|stts|target|tgt|method|algorithm|repository|repstry|commit|signif|signfcnt|risk|rsk|valid|data|sample|metric|feature|признак|статус|таргет|метод|репозитор|коммит|значим|риск|валидац|данн|выборк|метрик)'

spark.conf.set('spark.sql.session.timeZone', 'Europe/Moscow')
print('Конфигурация загружена. Источник:', CHANGE_TABLE)

## 1. Чтение и безопасная нормализация

Удаляются только технически удалённые записи (`ctl_action = 'D'`). Ограничение по `start_dttm`/`end_dttm` не применяется: для журнала это время изменения, а не интервал актуальности карточки.

In [ ]:
def require_table(table_name):
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError('Не найдена таблица: ' + table_name)

def require_columns(df, table_name, columns):
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise RuntimeError('{}: отсутствуют обязательные поля: {}'.format(table_name, ', '.join(missing)))

def clean_text(c):
    x = F.trim(c.cast('string'))
    return F.when(F.lower(x).isin('', 'null', 'none', 'n/a', 'na'), F.lit(None)).otherwise(x)

def parse_ts(c):
    x = F.trim(c.cast('string'))
    return F.coalesce(
        c.cast('timestamp'),
        F.to_timestamp(x, 'dd.MM.yyyy H:mm:ss'),
        F.to_timestamp(x, 'dd.MM.yyyy HH:mm:ss'),
        F.to_timestamp(x, 'yyyy-MM-dd HH:mm:ss.SSSSSS'),
        F.to_timestamp(x, 'yyyy-MM-dd HH:mm:ss'),
        F.to_timestamp(x, 'yyyy-MM-dd')
    )

require_table(CHANGE_TABLE)
src = spark.table(CHANGE_TABLE)
required = [
    'ent_param_chg_sid', 'ent_sid', 'ent_type_name',
    'ent_param_chg_name', 'ent_param_chg_prev_val', 'ent_param_chg_val',
    'ent_param_chg_type_code', 'ent_param_chg_usr_sid',
    'ent_param_chg_usr_name', 'start_dttm', 'end_dttm'
]
require_columns(src, CHANGE_TABLE, required)

if 'ctl_action' in src.columns:
    src = src.filter(F.coalesce(F.upper(F.trim(F.col('ctl_action'))), F.lit('')) != 'D')

log_df = src.select(
    F.col('ent_param_chg_sid').cast('string').alias('change_sid'),
    F.col('ent_sid').cast('string').alias('entity_sid'),
    F.upper(clean_text(F.col('ent_type_name'))).alias('entity_type'),
    clean_text(F.col('ent_param_chg_name')).alias('parameter_name'),
    clean_text(F.col('ent_param_chg_prev_val')).alias('previous_value'),
    clean_text(F.col('ent_param_chg_val')).alias('new_value'),
    F.upper(clean_text(F.col('ent_param_chg_type_code'))).alias('change_type'),
    parse_ts(F.col('start_dttm')).alias('change_dttm'),
    parse_ts(F.col('end_dttm')).alias('change_end_dttm'),
    F.xxhash64(
        F.coalesce(clean_text(F.col('ent_param_chg_usr_sid')), F.lit('$NULL$')),
        F.coalesce(clean_text(F.col('ent_param_chg_usr_name')), F.lit('$NULL$'))
    ).cast('string').alias('user_key'),
    (clean_text(F.col('ent_param_chg_usr_sid')).isNull() &
     clean_text(F.col('ent_param_chg_usr_name')).isNull()).alias('missing_user_flag')
).withColumn(
    'critical_parameter_flag',
    F.coalesce(F.col('parameter_name').rlike(CRITICAL_PARAM_RX), F.lit(False))
).persist(StorageLevel.MEMORY_AND_DISK)

total_changes = log_df.count()
print('Строк журнала после ctl_action-фильтра:', total_changes)
if total_changes == 0:
    raise RuntimeError('Журнал пуст: анализ невозможен')

## 2. Фактический состав журнала

In [ ]:
print('Типы сущностей:')
entity_type_stats = (log_df
    .groupBy(F.coalesce(F.col('entity_type'), F.lit('$NULL$')).alias('entity_type'))
    .agg(
        F.count('*').alias('change_count'),
        F.countDistinct('entity_sid').alias('entity_count'),
        F.countDistinct('parameter_name').alias('parameter_count'),
        F.min('change_dttm').alias('first_change_dttm'),
        F.max('change_dttm').alias('last_change_dttm')
    )
    .withColumn('share_pct', F.round(100.0 * F.col('change_count') / F.lit(total_changes), 2))
    .orderBy(F.desc('change_count')))
entity_type_stats.show(100, truncate=False)

print('Наиболее часто изменяемые параметры:')
parameter_stats = (log_df
    .groupBy('entity_type', 'parameter_name')
    .agg(
        F.count('*').alias('change_count'),
        F.countDistinct('entity_sid').alias('entity_count'),
        F.countDistinct('user_key').alias('user_count')
    )
    .orderBy(F.desc('change_count')))
parameter_stats.show(100, truncate=False)

## 3. Качество журнала и технические аномалии

In [ ]:
event_key_cols = [
    'entity_sid', 'entity_type', 'parameter_name',
    'previous_value', 'new_value', 'change_dttm', 'user_key'
]
dup_w = Window.partitionBy(*event_key_cols)
quality_df = (log_df
    .withColumn('duplicate_event_flag', F.count('*').over(dup_w) > 1)
    .withColumn('same_value_flag',
        F.col('previous_value').isNotNull() &
        F.col('new_value').isNotNull() &
        (F.col('previous_value') == F.col('new_value')))
    .withColumn('invalid_interval_flag',
        F.col('change_dttm').isNotNull() &
        F.col('change_end_dttm').isNotNull() &
        (F.col('change_end_dttm') < F.col('change_dttm')))
    .persist(StorageLevel.MEMORY_AND_DISK))

quality_summary = quality_df.agg(
    F.count('*').alias('row_count'),
    F.sum(F.col('entity_sid').isNull().cast('long')).alias('missing_entity_sid'),
    F.sum(F.col('entity_type').isNull().cast('long')).alias('missing_entity_type'),
    F.sum(F.col('parameter_name').isNull().cast('long')).alias('missing_parameter'),
    F.sum(F.col('previous_value').isNull().cast('long')).alias('missing_previous_value'),
    F.sum(F.col('new_value').isNull().cast('long')).alias('missing_new_value'),
    F.sum(F.col('change_dttm').isNull().cast('long')).alias('missing_change_dttm'),
    F.sum(F.col('missing_user_flag').cast('long')).alias('missing_user'),
    F.sum(F.col('same_value_flag').cast('long')).alias('same_value_events'),
    F.sum(F.col('invalid_interval_flag').cast('long')).alias('invalid_intervals'),
    F.sum(F.col('duplicate_event_flag').cast('long')).alias('duplicate_event_rows')
)
quality_summary.show(truncate=False)

print('Примеры технических аномалий без ФИО:')
quality_df.filter(
    F.col('same_value_flag') | F.col('invalid_interval_flag') | F.col('duplicate_event_flag') |
    F.col('entity_sid').isNull() | F.col('parameter_name').isNull() | F.col('change_dttm').isNull()
).select(
    'change_sid', 'entity_sid', 'entity_type', 'parameter_name',
    'previous_value', 'new_value', 'change_type', 'change_dttm',
    'same_value_flag', 'invalid_interval_flag', 'duplicate_event_flag'
).show(100, truncate=120)

## 4. Проверка связи с версиями моделей

Связь `entity_sid = model_ver_sid` проверяется фактически. Она не считается доказанной только по названию `entity_type`.

In [ ]:
require_table(MODEL_VER_TABLE)
mv_src = spark.table(MODEL_VER_TABLE)
require_columns(mv_src, MODEL_VER_TABLE, ['model_ver_sid', 'model_sid'])
if 'ctl_action' in mv_src.columns:
    mv_src = mv_src.filter(F.coalesce(F.upper(F.trim(F.col('ctl_action'))), F.lit('')) != 'D')

mv = mv_src.select(
    F.col('model_ver_sid').cast('string').alias('model_ver_sid'),
    F.col('model_sid').cast('string').alias('model_sid'),
    *(
        [F.col(c).cast('string').alias(c) for c in [
            'model_ver_stts_name', 'model_ver_signfcnt_ctgry_code',
            'model_ver_owner_dprtmt_name', 'model_ver_dev_dprtmt_name'
        ] if c in mv_src.columns]
    )
).dropDuplicates(['model_ver_sid']).persist(StorageLevel.MEMORY_AND_DISK)

version_linked = (quality_df.alias('c')
    .join(mv.alias('v'), F.col('c.entity_sid') == F.col('v.model_ver_sid'), 'left')
    .withColumn('direct_model_ver_match_flag', F.col('v.model_ver_sid').isNotNull())
    .persist(StorageLevel.MEMORY_AND_DISK))

linkage_stats = (version_linked
    .groupBy(F.coalesce(F.col('entity_type'), F.lit('$NULL$')).alias('entity_type'))
    .agg(
        F.count('*').alias('change_count'),
        F.sum(F.col('direct_model_ver_match_flag').cast('long')).alias('matched_change_count'),
        F.countDistinct(F.when(F.col('direct_model_ver_match_flag'), F.col('entity_sid'))).alias('matched_model_ver_count')
    )
    .withColumn('match_pct', F.round(100.0 * F.col('matched_change_count') / F.col('change_count'), 2))
    .orderBy(F.desc('matched_change_count')))
linkage_stats.show(100, truncate=False)

## 5. Активность, всплески и возвраты значений

In [ ]:
version_changes = version_linked.filter(F.col('direct_model_ver_match_flag'))

version_activity = (version_changes
    .groupBy('model_ver_sid')
    .agg(
        F.count('*').alias('change_count'),
        F.countDistinct('parameter_name').alias('parameter_count'),
        F.countDistinct(F.when(~F.col('missing_user_flag'), F.col('user_key'))).alias('user_count'),
        F.sum(F.col('critical_parameter_flag').cast('long')).alias('critical_change_count'),
        F.sum((F.col('change_dttm') >= F.date_sub(F.current_timestamp(), RECENT_DAYS)).cast('long')).alias('changes_last_30d'),
        F.min('change_dttm').alias('first_change_dttm'),
        F.max('change_dttm').alias('last_change_dttm')
    )
    .orderBy(F.desc('critical_change_count'), F.desc('change_count')))
print('Версии с наибольшей активностью:')
version_activity.show(100, truncate=False)

bursts = (version_changes
    .filter(F.col('change_dttm').isNotNull())
    .withColumn('change_hour', F.date_trunc('hour', F.col('change_dttm')))
    .groupBy('model_ver_sid', 'change_hour')
    .agg(
        F.count('*').alias('changes_in_hour'),
        F.countDistinct('parameter_name').alias('parameters_in_hour'),
        F.countDistinct('user_key').alias('users_in_hour')
    )
    .filter(F.col('changes_in_hour') >= BURST_CHANGES_PER_HOUR)
    .orderBy(F.desc('changes_in_hour')))
print('Всплески изменений:')
bursts.show(100, truncate=False)

seq_w = Window.partitionBy('entity_sid', 'parameter_name').orderBy('change_dttm', 'change_sid')
reversals = (version_changes
    .filter(F.col('change_dttm').isNotNull() & F.col('parameter_name').isNotNull())
    .withColumn('value_lag_1', F.lag('new_value', 1).over(seq_w))
    .withColumn('value_lag_2', F.lag('new_value', 2).over(seq_w))
    .filter(
        F.col('new_value').isNotNull() & F.col('value_lag_1').isNotNull() & F.col('value_lag_2').isNotNull() &
        (F.col('new_value') == F.col('value_lag_2')) & (F.col('new_value') != F.col('value_lag_1'))
    )
    .select('model_ver_sid', 'parameter_name', 'change_dttm', 'value_lag_2', 'value_lag_1', 'new_value', 'user_key')
    .orderBy(F.desc('change_dttm')))
print('Возвраты A → B → A:')
reversals.show(100, truncate=120)

## 6. Изменения после завершённой валидации

In [ ]:
require_table(VALID_TABLE)
valid_src = spark.table(VALID_TABLE)
require_columns(valid_src, VALID_TABLE, ['model_ver_sid', 'valid_stts_name', 'valid_end_fact_dttm'])
if 'ctl_action' in valid_src.columns:
    valid_src = valid_src.filter(F.coalesce(F.upper(F.trim(F.col('ctl_action'))), F.lit('')) != 'D')

completed_valid = (valid_src
    .select(
        F.col('model_ver_sid').cast('string').alias('valid_model_ver_sid'),
        F.upper(F.trim(F.col('valid_stts_name').cast('string'))).alias('valid_status'),
        parse_ts(F.col('valid_end_fact_dttm')).alias('valid_end_dttm')
    )
    .filter(
        F.col('valid_model_ver_sid').isNotNull() & F.col('valid_end_dttm').isNotNull() &
        F.col('valid_status').rlike(r'(?i)(DONE|NEGATIVE|COMPLET|FINISH|ЗАВЕРШ)')
    )
    .groupBy('valid_model_ver_sid')
    .agg(F.max('valid_end_dttm').alias('latest_valid_end_dttm')))

after_validation = (version_changes.alias('c')
    .join(completed_valid.alias('v'), F.col('c.model_ver_sid') == F.col('v.valid_model_ver_sid'), 'inner')
    .filter(F.col('c.change_dttm') > F.col('v.latest_valid_end_dttm'))
    .withColumn(
        'hours_after_validation',
        F.round((F.col('c.change_dttm').cast('long') - F.col('v.latest_valid_end_dttm').cast('long')) / 3600.0, 2)
    )
    .withColumn('within_24h_flag', F.col('hours_after_validation') <= F.lit(POST_VALIDATION_HOURS))
    .select(
        F.col('c.model_ver_sid'), 'c.change_sid', 'c.parameter_name',
        'c.previous_value', 'c.new_value', 'c.change_dttm',
        'v.latest_valid_end_dttm', 'hours_after_validation', 'within_24h_flag',
        'c.critical_parameter_flag', 'c.user_key'
    )
    .orderBy(F.desc('critical_parameter_flag'), F.asc('hours_after_validation')))

print('Изменения после последней завершённой валидации:')
after_validation.show(100, truncate=120)

## 7. Компактная оценка реализуемости контролей

Статусы означают техническую реализуемость по фактическим данным, а не наличие нарушения.

In [ ]:
base_metrics = version_linked.agg(
    F.count('*').cast('double').alias('n'),
    F.sum(F.col('direct_model_ver_match_flag').cast('double')).alias('matched'),
    F.sum(F.col('change_dttm').isNotNull().cast('double')).alias('with_date'),
    F.sum((~F.col('missing_user_flag')).cast('double')).alias('with_user'),
    F.sum((F.col('previous_value').isNotNull() | F.col('new_value').isNotNull()).cast('double')).alias('with_values'),
    F.sum(F.col('critical_parameter_flag').cast('double')).alias('critical')
)

metric_long = base_metrics.selectExpr(
    "stack(6, 'DIRECT_MODEL_VERSION_LINK', matched, n, 'CHANGE_TIMESTAMP_PRESENT', with_date, n, 'AUTHOR_PRESENT_HASHED', with_user, n, 'OLD_OR_NEW_VALUE_PRESENT', with_values, n, 'CRITICAL_PARAMETER_EVENTS', critical, n, 'SOURCE_NOT_EMPTY', n, n) as (check_name, numerator, denominator)"
).withColumn(
    'metric_pct', F.round(100.0 * F.col('numerator') / F.col('denominator'), 2)
)

control_readiness = metric_long.select(
    'check_name', 'numerator', 'denominator', 'metric_pct',
    F.when(F.col('check_name') == 'SOURCE_NOT_EMPTY',
        F.when(F.col('numerator') > 0, 'REALIZABLE').otherwise('NOT_REALIZABLE'))
     .when(F.col('check_name') == 'CRITICAL_PARAMETER_EVENTS',
        F.when(F.col('numerator') > 0, 'REALIZABLE').otherwise('NOT_CONFIRMED'))
     .when(F.col('metric_pct') >= 80, 'REALIZABLE')
     .when(F.col('metric_pct') >= 20, 'PARTIAL')
     .otherwise('NOT_REALIZABLE').alias('readiness_status')
).orderBy('check_name')
control_readiness.show(100, truncate=False)

overall = control_readiness.agg(
    F.sum((F.col('readiness_status') == 'REALIZABLE').cast('int')).alias('realizable_checks'),
    F.sum((F.col('readiness_status') == 'PARTIAL').cast('int')).alias('partial_checks'),
    F.sum(F.col('readiness_status').isin('NOT_REALIZABLE', 'NOT_CONFIRMED').cast('int')).alias('not_realizable_checks')
)
overall.show(truncate=False)

## 8. Опциональная запись обезличенных результатов

По умолчанию запись отключена. При включении сохраняются только агрегаты и обезличенные события; ФИО и исходный пользовательский SID не выгружаются.

In [ ]:
if WRITE_RESULTS:
    profile_out = (control_readiness
        .withColumn('source_table', F.lit(CHANGE_TABLE))
        .withColumn('as_of_dt', F.current_date())
        .withColumn('generated_dttm', F.current_timestamp()))

    details_out = after_validation.select(
        'model_ver_sid', 'change_sid', 'parameter_name',
        'previous_value', 'new_value', 'change_dttm',
        'latest_valid_end_dttm', 'hours_after_validation',
        'within_24h_flag', 'critical_parameter_flag', 'user_key'
    ).withColumn('as_of_dt', F.current_date())

    profile_out.repartition(1).write.mode('overwrite').format('parquet').saveAsTable(OUTPUT_PROFILE)
    details_out.repartition(4, 'model_ver_sid').write.mode('overwrite').format('parquet').saveAsTable(OUTPUT_DETAILS)
    print('Записано:', OUTPUT_PROFILE, OUTPUT_DETAILS)
else:
    print('WRITE_RESULTS=False: таблицы не записывались')

quality_df.unpersist()
version_linked.unpersist()
mv.unpersist()
log_df.unpersist()
print('Анализ завершён')